# ERC-20 Token Transfer Ağı — 01: Veri toplama ve temel istatistikler

**Sosyal Ağ Analizi · Yakup Kerim Tilki · 2516016032**

Bu defter ara kontrol için şunları üretir:

1. **Toplanmış veri:** Google BigQuery `crypto_ethereum.token_transfers` tablosundan tek bir token'ın bir haftalık transferleri
2. **Kenar listesi:** `data/edges.csv` (yönlü, ağırlıklı, anonimleştirilmiş) + `data/network.graphml`
3. **Temel istatistikler:** düğüm, kenar, yoğunluk, bileşenler → `results/temel_istatistikler.md`

**Nasıl çalıştırılır (Google Colab):** Üstteki *Çalışma zamanı → Tümünü çalıştır*. İlk hücrede `PROJECT_ID` değerini kendi Google Cloud proje kimliğinizle değiştirin. Google hesabıyla giriş izni ve "tuz" (salt) sorulacaktır.

**Ağ tanımı**
- **Düğüm:** Ethereum cüzdan/sözleşme adresi (tuzlu HMAC-SHA256 ile anonimleştirilmiş, 12 karakter)
- **Kenar:** A → B, haftada A'dan B'ye en az bir transfer varsa (yönlü)
- **Ağırlık:** toplam transfer miktarı (token birimi); ek öznitelik olarak transfer sayısı
- **Çıkarılanlar:** `0x000…000` adresi (mint/burn — gerçek bir cüzdan değil) ve kendine transferler (self-loop)

In [1]:
# --- AYARLAR -----------------------------------------------------------------
PROJECT_ID = "sna-erc20"

# Aday token'lar (Ethereum sözleşme adresi, ondalık basamak). Varsayılan: LINK.
TOKENS = {
    "LINK": ("0x514910771af9ca656af840dff83e8264ecf986ca", 18),
    "UNI":  ("0x1f9840a85d5af5bf1d1762f925bdaddc4201f984", 18),
    "AAVE": ("0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9", 18),
    "MKR":  ("0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2", 18),
    "SHIB": ("0x95ad61b0a150d79219dcf64e1e6cc01f0b64c4ce", 18),
}
TOKEN = "LINK"          # aday tablosuna bakıp gerekirse değiştirin
DAYS = 7                # bir haftalık pencere
SEED = 42               # sabit rastgele tohum (sonraki analizler için)
MAX_BYTES_BILLED = 60 * 10**9   # güvenlik sınırı: tek sorgu en fazla 60 GB tarasın (ücretsiz kota 1 TB/ay)
ZERO_ADDRESS = "0x0000000000000000000000000000000000000000"
TABLE = "bigquery-public-data.crypto_ethereum.token_transfers"

In [2]:
import os, json, hmac, hashlib, getpass, datetime as dt
import pandas as pd
import networkx as nx
import random
import numpy as np

random.seed(SEED); np.random.seed(SEED)
for d in ("data", "results", "private"):
    os.makedirs(d, exist_ok=True)
print("networkx", nx.__version__, "| pandas", pd.__version__)

networkx 3.7 | pandas 2.2.3


## 1. BigQuery bağlantısı

In [3]:
from google.cloud import bigquery
try:
    from google.colab import auth   # Colab'da Google hesabıyla giriş
    auth.authenticate_user()
except ImportError:
    pass  # yerelde: `gcloud auth application-default login` yapılmış olmalı

client = bigquery.Client(project=PROJECT_ID)

def run(sql, params=None, label=""):
    # Önce kuru çalıştırma ile taranacak veri miktarını göster, sonra güvenlik sınırıyla çalıştır.
    qp = [bigquery.ScalarQueryParameter(k, t, v) for k, (t, v) in (params or {}).items()]
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=qp))
    print(f"[{label}] taranacak veri: {dry.total_bytes_processed/1e9:.2f} GB")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED, query_parameters=qp))
    return job.to_dataframe()

## 2. Zaman penceresi: verideki en güncel tam hafta

In [4]:
latest = run(f'''
  SELECT MAX(block_timestamp) AS mx FROM `{TABLE}`
  WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 21 DAY)
''', label="son tarih")["mx"].iloc[0]

if pd.isna(latest):   # tablo son 3 haftada güncellenmediyse daha geniş bak
    latest = run(f'''
      SELECT MAX(block_timestamp) AS mx FROM `{TABLE}`
      WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 400 DAY)
    ''', label="son tarih (geniş)")["mx"].iloc[0]

END = pd.Timestamp(latest).tz_convert("UTC").normalize()      # son tam günün bitişi (00:00 UTC)
START = END - pd.Timedelta(days=DAYS)
print("Pencere (UTC):", START, "→", END)

[son tarih] taranacak veri: 1.49 GB
Pencere (UTC): 2026-10-19 00:00:00+00:00 → 2026-10-26 00:00:00+00:00


## 3. Aday token'ların haftalık hacmi (token seçimi için)

In [5]:
addr_list = ", ".join(f"'{a}'" for a, _ in TOKENS.values())
cand = run(f'''
  SELECT token_address,
         COUNT(*) AS transfer_sayisi,
         COUNT(DISTINCT from_address) + COUNT(DISTINCT to_address) AS adres_ust_sinir
  FROM `{TABLE}`
  WHERE block_timestamp >= @start AND block_timestamp < @end
    AND token_address IN ({addr_list})
  GROUP BY token_address
''', {"start": ("TIMESTAMP", START.to_pydatetime()), "end": ("TIMESTAMP", END.to_pydatetime())}, label="adaylar")
name_of = {a: n for n, (a, _) in TOKENS.items()}
cand["token"] = cand["token_address"].map(name_of)
cand.sort_values("transfer_sayisi")

[adaylar] taranacak veri: 5.32 GB


,token_address,transfer_sayisi,adres_ust_sinir,token
4,0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2,3566,684,MKR
3,0x95ad61b0a150d79219dcf64e1e6cc01f0b64c4ce,24935,13992,SHIB
2,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,46812,7861,AAVE
1,0x514910771af9ca656af840dff83e8264ecf986ca,63719,26908,LINK
0,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,107902,19641,UNI


> **Token seçimi:** Ağ en az 500 düğüm içermeli. Motif sayımı ve 100+ rastgele ağ (null model) hesaplanabilsin diye
> haftada yaklaşık **5.000–150.000 transfer** olan bir token uygundur. Varsayılan LINK; tabloya göre gerekirse
> ilk hücrede `TOKEN` değerini değiştirip defteri yeniden çalıştırın. Seçim gerekçesini rapora yazın.

## 4. Transferleri çek (kaynak → hedef çiftine göre toplanmış)

In [6]:
token_address, decimals = TOKENS[TOKEN]
raw = run(f'''
  SELECT from_address, to_address,
         COUNT(*) AS n_transfers,
         CAST(SUM(SAFE_CAST(value AS BIGNUMERIC)) AS FLOAT64) / POW(10, {decimals}) AS amount,
         MIN(block_timestamp) AS first_ts,
         MAX(block_timestamp) AS last_ts
  FROM `{TABLE}`
  WHERE block_timestamp >= @start AND block_timestamp < @end
    AND token_address = @token
  GROUP BY from_address, to_address
''', {"start": ("TIMESTAMP", START.to_pydatetime()), "end": ("TIMESTAMP", END.to_pydatetime()),
      "token": ("STRING", token_address)}, label="transferler")

raw["from_address"] = raw["from_address"].str.lower()
raw["to_address"] = raw["to_address"].str.lower()
raw.to_csv(f"private/ham_kenarlar_{TOKEN}.csv", index=False)   # ham adresli dosya: DEPOYA KONMAZ
print(f"{TOKEN}: {len(raw):,} adres çifti, {raw['n_transfers'].sum():,} transfer")
raw.head()

[transferler] taranacak veri: 5.78 GB
LINK: 28,033 adres çifti, 63,719 transfer


,from_address,to_address,n_transfers,amount,first_ts,last_ts
0,0x2338f069c81cae37472cea50d20fa3311cf4e933,0x77134cbc06cb00b66f4c7e623d5fdbf6777635ec,10,7312.276000,2026-10-19 18:13:23+00:00,2026-10-25 02:13:23+00:00
1,0xbeb0009aca35087ce7ccf11637e24dd1aad3bf2a,0x2e69797d37e888d62974a1ffa20816a5f55f898d,26,1729.488824,2026-10-19 00:46:47+00:00,2026-10-25 12:57:11+00:00
2,0xfad57d2039c21811c8f2b5d5b65308aa99d31559,0xbdb3ba9ffe392549e1f8658dd2630c141fdf47b6,323,17200.977844,2026-10-19 00:04:23+00:00,2026-10-25 23:34:59+00:00
3,0x000000000004444c5dc75cb358380d2e3de08a90,0xeff6cb8b614999d130e537751ee99724d01aa167,187,3007.557345,2026-10-19 00:43:35+00:00,2026-10-25 21:37:11+00:00
4,0x3d3f20583c80582b5db10724b8513e7fbc28b718,0x79dd9327352bcf1dd16941899705e97b61f4decb,12,77268.200000,2026-10-20 01:42:59+00:00,2026-10-25 06:53:59+00:00


## 5. Temizleme ve anonimleştirme

In [8]:
is_mint = raw["from_address"] == ZERO_ADDRESS
is_burn = raw["to_address"] == ZERO_ADDRESS
is_self = raw["from_address"] == raw["to_address"]
cleaning = {
    "mint_transfer": int(raw.loc[is_mint, "n_transfers"].sum()),
    "burn_transfer": int(raw.loc[is_burn, "n_transfers"].sum()),
    "self_loop_transfer": int(raw.loc[is_self & ~is_mint, "n_transfers"].sum()),
}
edges_raw = raw[~is_mint & ~is_burn & ~is_self].copy()
print("Çıkarılan transferler:", cleaning)

# Tuzlu HMAC-SHA256: tuz gizli tutulur (depoya konmaz), aynı tuzla tekrar çalıştırılınca aynı kimlikler üretilir.
SALT = os.environ.get("SNA_SALT") or getpass.getpass("Anonimleştirme tuzu (gizli, bir yere not edin): ")
assert len(SALT) >= 12, "Tuz en az 12 karakter olmalı"

def anon(addr):
    return "w_" + hmac.new(SALT.encode(), addr.encode(), hashlib.sha256).hexdigest()[:12]

addresses = pd.unique(pd.concat([edges_raw["from_address"], edges_raw["to_address"]]))
mapping = pd.DataFrame({"address": addresses})
mapping["node_id"] = mapping["address"].map(anon)
assert mapping["node_id"].is_unique, "Hash çakışması — daha uzun kimlik kullanın"
mapping.to_csv("private/adres_eslestirme.csv", index=False)   # borsa etiketleri için gerekli, DEPOYA KONMAZ
amap = dict(zip(mapping["address"], mapping["node_id"]))

edges = pd.DataFrame({
    "source": edges_raw["from_address"].map(amap),
    "target": edges_raw["to_address"].map(amap),
    "weight": edges_raw["amount"].round(6),
    "n_transfers": edges_raw["n_transfers"].astype(int),
    "first_ts": pd.to_datetime(edges_raw["first_ts"], utc=True).dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "last_ts": pd.to_datetime(edges_raw["last_ts"], utc=True).dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
}).sort_values(["source", "target"]).reset_index(drop=True)
edges.head()

Çıkarılan transferler: {'mint_transfer': 0, 'burn_transfer': 0, 'self_loop_transfer': 22}
Anonimleştirme tuzu (gizli, bir yere not edin): ··········


,source,target,weight,n_transfers,first_ts,last_ts
0,w_000155d93836,w_a53229ef63ee,0.000200,2,2026-10-22T15:57:47Z,2026-10-25T23:23:59Z
1,w_00041aae7c33,w_a753b89ed4ea,3.901410,1,2026-10-23T11:19:23Z,2026-10-23T11:19:23Z
2,w_000b15739d75,w_195f6239fc9f,33.130719,3,2026-10-21T21:40:35Z,2026-10-21T22:28:59Z
3,w_000b86e9de10,w_2bc867731c46,10692.127845,3,2026-10-22T09:58:23Z,2026-10-25T12:28:35Z
4,w_000b86e9de10,w_429e6d43a774,3243.959398,2,2026-10-20T13:09:11Z,2026-10-21T14:22:11Z


## 6. Ağı kur

In [9]:
G = nx.DiGraph()
G.add_weighted_edges_from(edges[["source", "target", "weight"]].itertuples(index=False, name=None))
nx.set_edge_attributes(G, {(s, t): n for s, t, n in edges[["source", "target", "n_transfers"]].itertuples(index=False, name=None)}, "n_transfers")

out_amt = edges.groupby("source")["weight"].sum()
in_amt = edges.groupby("target")["weight"].sum()
out_n = edges.groupby("source")["n_transfers"].sum()
in_n = edges.groupby("target")["n_transfers"].sum()
nodes = pd.DataFrame({"node_id": list(G.nodes())})
nodes["in_degree"] = nodes["node_id"].map(dict(G.in_degree()))
nodes["out_degree"] = nodes["node_id"].map(dict(G.out_degree()))
nodes["sent_amount"] = nodes["node_id"].map(out_amt).fillna(0).round(6)
nodes["received_amount"] = nodes["node_id"].map(in_amt).fillna(0).round(6)
nodes["sent_transfers"] = nodes["node_id"].map(out_n).fillna(0).astype(int)
nodes["received_transfers"] = nodes["node_id"].map(in_n).fillna(0).astype(int)
for col in ["in_degree", "out_degree", "sent_amount", "received_amount", "sent_transfers", "received_transfers"]:
    nx.set_node_attributes(G, dict(zip(nodes["node_id"], nodes[col].tolist())), col)
print(G)

DiGraph with 20348 nodes and 28025 edges


## 7. Temel istatistikler

In [10]:
n, m = G.number_of_nodes(), G.number_of_edges()
wcc = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
scc = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
stats = {
    "token": TOKEN,
    "token_adresi": token_address,
    "pencere_utc": f"{START:%Y-%m-%d} – {END - pd.Timedelta(seconds=1):%Y-%m-%d}",
    "toplam_transfer_ham": int(raw["n_transfers"].sum()),
    "agdaki_transfer": int(edges["n_transfers"].sum()),
    "cikarilan": cleaning,
    "toplam_hacim_token": round(float(edges["weight"].sum()), 2),
    "dugum_sayisi": n,
    "kenar_sayisi": m,
    "yogunluk": nx.density(G),
    "ortalama_derece_giris_cikis": m / n,
    "max_giris_derecesi": int(nodes["in_degree"].max()),
    "max_cikis_derecesi": int(nodes["out_degree"].max()),
    "karsiliklilik_reciprocity": nx.reciprocity(G),
    "zayif_bilesen_sayisi": len(wcc),
    "en_buyuk_zayif_bilesen": len(wcc[0]),
    "en_buyuk_zayif_bilesen_orani": len(wcc[0]) / n,
    "guclu_bilesen_sayisi": len(scc),
    "en_buyuk_guclu_bilesen": len(scc[0]),
    "tek_dugumlu_guclu_bilesen": sum(1 for c in scc if len(c) == 1),
    "zayif_bilesen_boyut_dagilimi_ilk10": [len(c) for c in wcc[:10]],
}
if n < 500:
    print("UYARI: ağ 500 düğümden küçük — daha yoğun bir token seçin.")
pd.Series({k: v for k, v in stats.items() if not isinstance(v, (dict, list))}).to_frame("değer")

,değer
token,LINK
token_adresi,0x514910771af9ca656af840dff83e8264ecf986ca
pencere_utc,2026-10-19 – 2026-10-25
toplam_transfer_ham,63719
agdaki_transfer,63697
toplam_hacim_token,39248100.64
dugum_sayisi,20348
kenar_sayisi,28025
yogunluk,0.000068
ortalama_derece_giris_cikis,1.377285


## 8. Dosyaları kaydet

In [11]:
edges.to_csv("data/edges.csv", index=False)
nodes.to_csv("data/nodes.csv", index=False)
nx.write_graphml(G, "data/network.graphml")
with open("results/temel_istatistikler.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2, default=float)

def fmt(x):
    if isinstance(x, float):
        return f"{x:.6g}" if abs(x) < 1e-3 else f"{x:,.4f}"
    if isinstance(x, int):
        return f"{x:,}"
    return str(x)

rows = [
    ("Token", f"{TOKEN} ({token_address})"),
    ("Zaman penceresi (UTC)", stats["pencere_utc"]),
    ("Toplam transfer (ham)", fmt(stats["toplam_transfer_ham"])),
    ("Ağdaki transfer (temizlik sonrası)", fmt(stats["agdaki_transfer"])),
    ("Çıkarılan: mint / burn / self-loop", f"{cleaning['mint_transfer']:,} / {cleaning['burn_transfer']:,} / {cleaning['self_loop_transfer']:,}"),
    ("Düğüm sayısı", fmt(n)),
    ("Kenar sayısı (yönlü)", fmt(m)),
    ("Yoğunluk", fmt(stats["yogunluk"])),
    ("Ortalama giriş (= çıkış) derecesi", fmt(stats["ortalama_derece_giris_cikis"])),
    ("En yüksek giriş / çıkış derecesi", f"{stats['max_giris_derecesi']:,} / {stats['max_cikis_derecesi']:,}"),
    ("Karşılıklılık (reciprocity)", fmt(stats["karsiliklilik_reciprocity"])),
    ("Zayıf bağlı bileşen sayısı", fmt(len(wcc))),
    ("En büyük zayıf bileşen", f"{len(wcc[0]):,} düğüm (%{100*len(wcc[0])/n:.1f})"),
    ("Güçlü bağlı bileşen sayısı", fmt(len(scc))),
    ("En büyük güçlü bileşen", f"{len(scc[0]):,} düğüm"),
    ("Toplam hacim", f"{stats['toplam_hacim_token']:,.2f} {TOKEN}"),
]
md_table = "| Ölçü | Değer |\n| --- | --- |\n" + "\n".join(f"| {a} | {b} |" for a, b in rows)
with open("results/temel_istatistikler.md", "w", encoding="utf-8") as f:
    f.write(f"# Temel istatistikler — {TOKEN} transfer ağı\n\n{md_table}\n")

with open("veri_sozlugu.md", "w", encoding="utf-8") as f:
    f.write(f'''# Veri sözlüğü

- **Kaynak:** Google BigQuery public dataset `bigquery-public-data.crypto_ethereum.token_transfers`
  (Ethereum ERC-20 `Transfer` olayları; Google Cloud Public Datasets, açık blokzincir verisi)
- **Token:** {TOKEN} — `{token_address}` ({decimals} ondalık)
- **Zaman penceresi (UTC):** {stats["pencere_utc"]}
- **Toplama tarihi:** {dt.datetime.now(dt.timezone.utc):%Y-%m-%d %H:%M} UTC
- **Anonimleştirme:** adresler gizli bir tuzla HMAC-SHA256 → ilk 12 hex karakter (`w_` önekli). Tuz ve
  adres eşleştirme tablosu depoda yayımlanmaz.

## data/edges.csv (kenar listesi)
| Sütun | Anlamı |
| --- | --- |
| source | Gönderen cüzdanın anonim kimliği |
| target | Alıcı cüzdanın anonim kimliği |
| weight | Pencere boyunca source→target toplam transfer miktarı ({TOKEN} birimi) |
| n_transfers | Pencere boyunca source→target transfer sayısı |
| first_ts / last_ts | İlk ve son transferin zamanı (UTC, ISO 8601) |

## data/nodes.csv (düğüm listesi)
| Sütun | Anlamı |
| --- | --- |
| node_id | Anonim cüzdan kimliği |
| in_degree / out_degree | Farklı kaç cüzdandan para aldığı / kaç cüzdana gönderdiği |
| sent_amount / received_amount | Toplam gönderilen / alınan miktar ({TOKEN}) |
| sent_transfers / received_transfers | Toplam gönderim / alım işlemi sayısı |

## data/network.graphml
Aynı yönlü ağırlıklı ağ; düğüm ve kenar öznitelikleri yukarıdaki sütunlarla aynıdır (Gephi ile açılabilir).
''')

print(open("results/temel_istatistikler.md", encoding="utf-8").read())
for p in ["data/edges.csv", "data/nodes.csv", "data/network.graphml"]:
    print(p, f"{os.path.getsize(p)/1e6:.1f} MB")

# Temel istatistikler — LINK transfer ağı

| Ölçü | Değer |
| --- | --- |
| Token | LINK (0x514910771af9ca656af840dff83e8264ecf986ca) |
| Zaman penceresi (UTC) | 2026-10-19 – 2026-10-25 |
| Toplam transfer (ham) | 63,719 |
| Ağdaki transfer (temizlik sonrası) | 63,697 |
| Çıkarılan: mint / burn / self-loop | 0 / 0 / 22 |
| Düğüm sayısı | 20,348 |
| Kenar sayısı (yönlü) | 28,025 |
| Yoğunluk | 6.76898e-05 |
| Ortalama giriş (= çıkış) derecesi | 1.3773 |
| En yüksek giriş / çıkış derecesi | 2,780 / 839 |
| Karşılıklılık (reciprocity) | 0.0784 |
| Zayıf bağlı bileşen sayısı | 579 |
| En büyük zayıf bileşen | 18,896 düğüm (%92.9) |
| Güçlü bağlı bileşen sayısı | 16,604 |
| En büyük güçlü bileşen | 3,698 düğüm |
| Toplam hacim | 39,248,100.64 LINK |

data/edges.csv 2.3 MB
data/nodes.csv 0.8 MB
data/network.graphml 7.5 MB


## 9. (Colab) Sonuçları indir

In [12]:
import shutil
# Yalnızca yayımlanacak dosyalar (private/ HARİÇ):
os.makedirs("yayin", exist_ok=True)
for p in ["data", "results"]:
    shutil.copytree(p, f"yayin/{p}", dirs_exist_ok=True)
shutil.copy("veri_sozlugu.md", "yayin/veri_sozlugu.md")
shutil.make_archive("yayin_dosyalari", "zip", "yayin")
try:
    from google.colab import files
    files.download("yayin_dosyalari.zip")
except ImportError:
    print("yayin_dosyalari.zip hazır")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Yorumlar (kendi cümlelerinizle doldurun)

- **Düğüm / kenar sayısı:** Bu hafta … farklı cüzdan … farklı yönde para akışıyla birbirine bağlı …
- **Yoğunluk:** Olası bağlantıların yalnızca … kadarı gerçekleşmiş; yani ağ … (seyrek mi?) Bu, transfer ağlarında …
- **Bileşenler:** Cüzdanların %… tek bir dev bileşende; bu, token'ın … üzerinden dolaştığını gösteriyor …
- **Güçlü bileşenler / karşılıklılık:** Paranın geri döndüğü (A→B ve B→A) ilişkiler …